In [1]:
import os
import numpy as np
import rubin_sim
import rubin_sim.maf as maf
from rubin_sim.data import get_baseline

# Obtener el path al archivo .db del baseline simulado
baseline_file = get_baseline()
conn = baseline_file  # usar directamente el path como string

outDir = 'temp'
resultsDb = maf.db.ResultsDb()  # <- CORREGIDO

# Coordenadas
Ra, Dec = 270, -30
ra = [Ra]
dec = [Dec]

# Métrica
metric = maf.metrics.PassMetric(cols=['filter', 'observationStartMJD', 'fiveSigmaDepth'])

# Slicer
slicer = maf.slicers.UserPointsSlicer(ra=ra, dec=dec)

# SQL vacío
sql = ''

# Crear MetricBundle
metric_bundle = maf.MetricBundle(metric, slicer, sql)
bundleDict = {'my_bundle': metric_bundle}

# Ejecutar
bg = maf.MetricBundleGroup(bundleDict, conn, out_dir=outDir, results_db=resultsDb)
bg.run_all()


# Ver resultado
dataSlice = metric_bundle.metric_values[0]
print("Data slice:", dataSlice)


Data slice: [('y', 268.87400589, 62754.12414483,  -8.26996137, 21.69349242, -28.60776889)
 ('y', 268.87400589, 62754.10129085,  -8.26996137, 21.70971516, -28.60776889)
 ('r', 268.66069603, 63041.0234675 , 197.72513443, 23.89843319, -28.55809045)
 ...
 ('g', 271.06264576, 63738.37042616, -19.41459364, 24.07856365, -29.39531689)
 ('g', 272.14197626, 64547.16304929,  10.96006464, 24.40836179, -29.57592946)
 ('r', 272.14197626, 64547.18718397,  10.96006464, 23.60075374, -29.57592946)]


In [7]:
{f: max(dataSlice[dataSlice['filter']==f]['fiveSigmaDepth']) for f in 'ugrizy'}

{'u': np.float64(23.84310878069171),
 'g': np.float64(25.85541372733996),
 'r': np.float64(24.869749376199714),
 'i': np.float64(24.293404515574974),
 'z': np.float64(23.70813344712104),
 'y': np.float64(22.52238790381355)}

In [11]:
{'a':1}|{'b':2}

{'a': 1, 'b': 2}

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm 
from scipy import stats
dfs = []
for i in range(1,11):
    data_split = pd.read_csv(f'../chunks_TRILEGAL_GENULENS/TRILEGAL_chunk_{i}.csv')
    dfs.append(data_split)
data = pd.concat(dfs, ignore_index=True)
data

,#Gc,logAge,[M/H],m_ini,logL,logTe,logg,m-M0,Av,m2/m1,...,z,Y,R062,Z087,Y106,J129,H158,F184,W149,Mact
0,1,6.65,-0.41,0.20008,-2.167,3.564,5.109,12.9,0.026,0.00,...,22.687,22.509,23.600,22.202,21.746,21.293,20.891,20.734,22.4888,0.200
1,1,6.65,-0.41,0.03868,-2.491,3.434,4.197,13.5,0.029,0.00,...,24.371,23.905,26.615,23.870,22.977,22.471,22.111,21.980,23.7048,0.039
2,1,6.65,-0.41,0.24626,-1.992,3.575,5.068,14.3,0.032,0.00,...,23.641,23.470,24.477,23.156,22.718,22.277,21.874,21.728,23.4708,0.246
3,1,6.65,-0.41,0.06566,-2.198,3.466,4.261,14.4,0.032,0.00,...,24.395,24.072,26.276,23.912,23.192,22.686,22.287,22.183,23.9078,0.066
4,1,6.65,-0.41,0.10981,-1.835,3.514,4.316,15.0,0.032,0.80,...,23.416,23.174,24.797,22.935,22.358,21.865,21.466,21.320,23.0758,0.110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,1,7.27,0.03,0.26887,-1.998,3.553,5.024,16.2,0.035,0.76,...,25.062,24.850,26.229,24.581,24.071,23.603,23.205,23.037,24.8028,0.269
99996,1,7.27,0.03,0.14832,-1.775,3.501,4.335,16.2,0.035,0.00,...,25.069,24.781,26.643,24.588,23.954,23.473,23.101,22.963,24.6958,0.148
99997,1,7.27,0.03,0.26188,-2.019,3.552,5.029,16.2,0.035,0.00,...,25.622,25.416,26.758,25.141,24.640,24.173,23.772,23.600,25.3698,0.262
99998,1,7.27,0.03,1.26571,0.354,3.807,4.359,16.2,0.035,0.00,...,19.978,19.989,19.887,19.485,19.371,19.162,18.989,18.950,20.4008,1.266


In [2]:

data['y'] = data['Y']
bands = ['W149','u','g','r','i','z','y']

mag_lim = {f: np.mean(dataSlice[dataSlice['filter']==f]['fiveSigmaDepth']) for f in 'ugrizy'}|{'W149':27.6}
mag_sat = {'W149': 14.8, 'u': 14.7, 'g': 15.7, 'r': 15.8, 'i': 15.8, 'z': 15.3, 'y': 13.9}

for f in bands:
    # if f =='W149':
    #     plt.hist(data[f], bins= np.linspace(mag_sat[f], mag_lim[f], 15),
                 # alpha=0.9,rwidth=0.8,edgecolor='k',linewidth=0.5,color='pink')
    if not f =='W149':
        plt.hist(data[f], bins= np.linspace(mag_sat[f], mag_lim[f], 15),
                 histtype='step',alpha=1,linewidth=3)


# plt.yscale('log')
plt.xlabel("mag")
# plt.xscale('log')

NameError: name 'dataSlice' is not defined